<a href="https://colab.research.google.com/github/BandanaSingha24/02_Genomics-end-to-end-Pipelines-for-Bioinformatics/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install tools (Conda is slow -> use apt + prebuilt binaries where possible)
!apt-get update && apt-get install -y samtools bwa fastqc tabix
!pip install cyvcf2 # for VCF reading later

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,945 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,070 kB]
Hit:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy In

In [5]:
# Install GATK (overwrite automatically)
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.5.0.0/gatk-4.5.0.0.zip
!unzip -o -q gatk-4.5.0.0.zip
!mv gatk-4.5.0.0/gatk /usr/local/bin/
!chmod +x /usr/local/bin/gatk
print("GATK installed successfully!")

GATK installed successfully!


In [29]:
!rm -rf vep_cache/

In [28]:
# Install Ensembl VEP (lightweight offline mode - needs cache)
!wget -q https://ftp.ensembl.org/pub/release-112/variation/indexed_vep_cache/homo_sapiens_112_GRCh38.tar.gz -O vep_cache.tar.gz
!mkdir -p vep_cache && tar -xzf vep_cache.tar.gz -C vep_cache
!rm vep_cache.tar.gz


gzip: stdin: unexpected end of file
tar: Child returned status 1
tar: Error is not recoverable: exiting now


In [24]:
# Clean up potential partial download first
!rm -f vep_cache.tar.gz
!rm -rf vep_cache

# Try downloading and extracting again
!wget -q https://ftp.ensembl.org/pub/release-112/variation/indexed_vep_cache/homo_sapiens_112_GRCh38.tar.gz -O vep_cache.tar.gz
!mkdir -p vep_cache && tar -xzf vep_cache.tar.gz -C vep_cache
!rm vep_cache.tar.gz

print("VEP cache setup attempt complete.")


gzip: stdin: unexpected end of file
tar: Child returned status 1
tar: Error is not recoverable: exiting now
VEP cache setup attempt complete.


In [ ]:
# Install essential bio-tools
!apt-get update && apt-get install -y openjdk-11-jdk-headless samtools bwa fastqc wget unzip bcftools


# Download and setup GATK 4.5.0.0
!wget -q https://github.com/broadinstitute/gatk/releases/download/4.5.0.0/gatk-4.5.0.0.zip
!unzip -o -q gatk-4.5.0.0.zip
!mv gatk-4.5.0.0 gatk_install

# Create wrapper script for easy execution
!echo '#!/bin/bash' > /usr/local/bin/gatk
!echo 'java -jar /content/gatk_install/gatk-package-4.5.0.0-local.jar "$@"' >> /usr/local/bin/gatk
!chmod +x /usr/local/bin/gatk

# Verify GATK installation
!gatk --version
print("GATK and BCFtools are ready for analysis!")

In [16]:
# Step 1: Download tumor-normal FASTQ files
import os
os.makedirs("data/fastq", exist_ok=True)

base = "https://zenodo.org/record/2582555/files/"
files = [
    "SLGFSK-N_231335_r1_chr5_12_17.fastq.gz",
    "SLGFSK-N_231335_r2_chr5_12_17.fastq.gz",
    "SLGFSK-T_231336_r1_chr5_12_17.fastq.gz",
    "SLGFSK-T_231336_r2_chr5_12_17.fastq.gz"
]
for f in files:
    !wget -nc {base + f} -O data/fastq/{f}

print("FASTQ files downloaded successfully!")
!ls -lh data/fastq/


--2026-05-21 18:39:25--  https://zenodo.org/record/2582555/files/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Resolving zenodo.org (zenodo.org)... 137.138.153.219, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|137.138.153.219|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/2582555/files/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz [following]
--2026-05-21 18:39:26--  https://zenodo.org/records/2582555/files/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 530427108 (506M) [application/octet-stream]
Saving to: ‘data/fastq/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz’

data/fastq/SLGFSK-N 100%[===================>] 505.85M  21.2MB/s    in 27s     

2026-05-21 18:39:53 (19.0 MB/s) - ‘data/fastq/SLGFSK-N_231335_r1_chr5_12_17.fastq.gz’ saved [530427108/530427108]

--2026-05-21 18:39:53--  https://zenodo.org/record/2582555/files/SLGFSK-N_23

In [17]:
# Step 2: FASTQ QC
# FASTQC - Quality Control
!mkdir -p results/fastqc
!fastqc -o results/fastqc -t 2 data/fastq/*.fastq.gz

Started analysis of SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Started analysis of SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 5% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 5% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 10% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 10% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 15% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 15% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 20% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 20% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 25% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 25% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 30% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 30% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Approx 35% complete for SLGFSK-N_231335_r1_chr5_12_17.fastq.gz
Approx 35% complete for SLGFSK-N_231335_r2_chr5_12_17.fastq.gz
Ap

In [18]:
# Step 3: Alignment → BAM with BWA-MEM

# FAST Subsampling + Quick Alignment
import os
os.makedirs("data/subsampled_fastq", exist_ok=True)
os.makedirs("results", exist_ok=True)

# 1. Download the pre-sliced reference (hg19 chr5_12_17 Mb slice)
!wget -nc https://zenodo.org/record/2582555/files/hg19.chr5_12_17.fa.gz
!gunzip -f hg19.chr5_12_17.fa.gz
!mv hg19.chr5_12_17.fa ref_chr5_slice.fa

# Index reference (very fast)
!bwa index ref_chr5_slice.fa
!samtools faidx ref_chr5_slice.fa

# 2. Subsample to first 100,000 read pairs per file
import os
os.makedirs("data/subsampled_fastq", exist_ok=True)
os.makedirs("results", exist_ok=True)

number_of_pairs = 50000
lines_to_take = number_of_pairs * 4

samples = ['N', 'T'] # N=normal, T=tumor
for sample in samples:
    for read in [1, 2]:
        input_file = f"data/fastq/SLGFSK-{sample}_23133{3 if sample=='T' else 2}_r{read}_chr5_12_17.fastq.gz"
        output_file = f"data/subsampled_fastq/SLGFSK-{sample}_sub_r{read}.fastq.gz"
        !zcat {input_file} | head -n {lines_to_take} | gzip > {output_file}

!ls -lh data/subsampled_fastq/

# 3. Align subsampled normal
!bwa mem -t 4 -R '@RG\tID:normal\tSM:normal\tLB:lib' ref_chr5_slice.fa \
    data/subsampled_fastq/SLGFSK-N_sub_r1.fastq.gz \
    data/subsampled_fastq/SLGFSK-N_sub_r2.fastq.gz | \
    samtools sort -o results/normal.bam -
!samtools index results/normal.bam

# Align subsampled tumor
!bwa mem -t 4 -R '@RG\tID:tumor\tSM:tumor\tLB:lib' ref_chr5_slice.fa \
    data/subsampled_fastq/SLGFSK-T_sub_r1.fastq.gz \
    data/subsampled_fastq/SLGFSK-T_sub_r2.fastq.gz | \
    samtools sort -o results/tumor.bam -
!samtools index results/tumor.bam

print("Subsampled BAM files ready!")
!ls -lh results/*.bam
!du -sh results/*.bam # shows human-readable sizes

--2026-05-21 19:12:26--  https://zenodo.org/record/2582555/files/hg19.chr5_12_17.fa.gz
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 188.185.43.153, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 301 MOVED PERMANENTLY
Location: /records/2582555/files/hg19.chr5_12_17.fa.gz [following]
--2026-05-21 19:12:26--  https://zenodo.org/records/2582555/files/hg19.chr5_12_17.fa.gz
Reusing existing connection to zenodo.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 126162910 (120M) [application/octet-stream]
Saving to: ‘hg19.chr5_12_17.fa.gz’

hg19.chr5_12_17.fa. 100%[===================>] 120.32M  12.6MB/s    in 97s     

2026-05-21 19:14:04 (1.24 MB/s) - ‘hg19.chr5_12_17.fa.gz’ saved [126162910/126162910]

[bwa_index] Pack FASTA... 4.16 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=791924730, availableWord=67722660
[BWTIncConstructFromPacked] 10 iteratio

In [19]:
# Step 4: Somatic Variant Calling with Mutect2

# Ensure reference is fully prepared for GATK
!samtools faidx ref_chr5_slice.fa
!gatk CreateSequenceDictionary -R ref_chr5_slice.fa -O ref_chr5_slice.dict

!ls -lh ref_chr5_slice.*  # show .fa, .fai, .dict, .amb/.ann/.bwt etc. from bwa

No local jar was found, please build one by running

    /usr/local/bin/gradlew localJar
or
    export GATK_LOCAL_JAR=<path_to_local_jar>
-rw-r--r-- 1 root root 386M May 21 19:14 ref_chr5_slice.fa
-rw-r--r-- 1 root root  485 May 21 19:21 ref_chr5_slice.fa.amb
-rw-r--r-- 1 root root  117 May 21 19:21 ref_chr5_slice.fa.ann
-rw-r--r-- 1 root root 378M May 21 19:21 ref_chr5_slice.fa.bwt
-rw-r--r-- 1 root root   86 May 21 19:32 ref_chr5_slice.fa.fai
-rw-r--r-- 1 root root  95M May 21 19:21 ref_chr5_slice.fa.pac
-rw-r--r-- 1 root root 189M May 21 19:25 ref_chr5_slice.fa.sa


In [21]:
# Somatic Variant Calling - GATK Mutect2

# Limit to the chr5 slice region for speed
interval = "chr5:12000000-17000000"

!gatk Mutect2 \
    -R ref_chr5_slice.fa \
    -I results/tumor.bam \
    -I results/normal.bam \
    -normal normal \
    -L {interval} \
    --max-mnp-distance 0 \
    -O results/raw_somatic.vcf.gz

# Quick stats (number of variants etc.)
!bcftools stats results/raw_somatic.vcf.gz | grep -E "number of|SNPs|indels"

print("Raw VCF ready! Check for candidate somatic variants.")
!ls -lh results/raw_somatic.vcf.gz

No local jar was found, please build one by running

    /usr/local/bin/gradlew localJar
or
    export GATK_LOCAL_JAR=<path_to_local_jar>
/bin/bash: line 1: bcftools: command not found
Raw VCF ready! Check for candidate somatic variants.
ls: cannot access 'results/raw_somatic.vcf.gz': No such file or directory
